# Introductory Statistics – Part 6: Regression and ANOVA

Welcome to Part 6 — the final instalment of this series. We tackle two of the most widely used tools in applied statistics: **regression** for modelling relationships between variables, and **ANOVA** for comparing means across multiple groups.

**By the end of this notebook you will be able to:**

- Fit and interpret a **simple linear regression** model
- Compute and interpret **Pearson** and **Spearman** correlation coefficients
- Conduct and interpret a **one-way ANOVA**
- Fit and interpret a **multiple regression** model

**Topics covered:**

21. Linear regression, least squares, correlation, Spearman rank correlation, inference for regression  
22. One-way ANOVA  
23. Multiple regression

> **Prerequisite:** Part 5 – Two-Sample Inference.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set(style="whitegrid")
np.random.seed(42)

---
## 21. Linear Regression and Correlation

### 21.1 Simple Linear Regression

Simple linear regression models the relationship between a **response variable** $y$ and a single **predictor** $x$:

$$y = \beta_0 + \beta_1 x + \varepsilon, \quad \varepsilon \sim N(0, \sigma^2)$$

| Parameter | Interpretation |
|---|---|
| $\beta_0$ | Intercept — predicted value of $y$ when $x = 0$ |
| $\beta_1$ | Slope — change in $y$ for a one-unit increase in $x$ |
| $\varepsilon$ | Random error (residual) |

### 21.2 Least Squares Estimation

The **ordinary least squares (OLS)** method finds $\hat{\beta}_0$ and $\hat{\beta}_1$ to minimise the **residual sum of squares**:

$$\text{RSS} = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 = \sum_{i=1}^{n}(y_i - \hat{\beta}_0 - \hat{\beta}_1 x_i)^2$$

The formulas are:

$$\hat{\beta}_1 = \frac{\sum(x_i - \bar{x})(y_i - \bar{y})}{\sum(x_i - \bar{x})^2}, \qquad \hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x}$$

In [ ]:
# Simulate data from y = 3 + 2x + ε
np.random.seed(42)
x = np.linspace(0, 10, 50)
y = 3 + 2*x + np.random.normal(scale=3, size=50)

df = pd.DataFrame({"x": x, "y": y})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatterplot with regression line
sns.scatterplot(data=df, x="x", y="y", ax=axes[0], alpha=0.7)
sns.regplot(data=df, x="x", y="y", scatter=False, color="red",
            ax=axes[0], label="OLS fit")
axes[0].set_title("Scatterplot with Regression Line")
axes[0].legend()

# Residuals vs fitted
model_prelim = smf.ols("y ~ x", data=df).fit()
axes[1].scatter(model_prelim.fittedvalues, model_prelim.resid,
                alpha=0.7, color="steelblue")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title("Residuals vs Fitted")
axes[1].set_xlabel("Fitted values $\\hat{y}$")
axes[1].set_ylabel("Residuals $e_i = y_i - \\hat{y}_i$")

plt.suptitle("Simple Linear Regression: $y = 3 + 2x + \\varepsilon$", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Fit and summarise the regression model using statsmodels
model = smf.ols("y ~ x", data=df).fit()

# Extract key results
print("=== Regression Results ===")
print(f"  Intercept  (β̂₀): {model.params['Intercept']:.3f}")
print(f"  Slope      (β̂₁): {model.params['x']:.3f}")
print(f"  R²:              {model.rsquared:.4f}")
print(f"  Adjusted R²:     {model.rsquared_adj:.4f}")
print()
print("Full summary:")
print(model.summary())

### 21.3 Correlation

Correlation measures the **strength and direction** of a linear (or monotonic) relationship.

| Measure | Formula | Range | Best for |
|---|---|---|---|
| **Pearson $r$** | $r = \dfrac{\sum(x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum(x_i-\bar{x})^2 \sum(y_i-\bar{y})^2}}$ | $[-1, 1]$ | Linear relationships, continuous data |
| **Spearman $r_s$** | Pearson $r$ applied to the **ranks** of $x$ and $y$ | $[-1, 1]$ | Monotonic relationships, ordinal data, or outliers |

Note: $r^2$ equals $R^2$ from simple linear regression — it tells you the **proportion of variance** in $y$ explained by $x$.

In [ ]:
# Pearson and Spearman correlations
pearson_r,  pearson_p  = pearsonr(df["x"], df["y"])
spearman_r, spearman_p = spearmanr(df["x"], df["y"])

print(f"Pearson  r  = {pearson_r:.4f},  p = {pearson_p:.4e}")
print(f"Spearman rₛ = {spearman_r:.4f},  p = {spearman_p:.4e}")
print(f"R² = r² = {pearson_r**2:.4f}  (matches model R²: {model.rsquared:.4f})")

---
## 22. One-Way ANOVA

**One-way ANOVA** (Analysis of Variance) tests whether the means of **three or more independent groups** are equal.

### Model

$$y_{ij} = \mu + \tau_i + \varepsilon_{ij}, \quad \varepsilon_{ij} \sim N(0, \sigma^2)$$

where $\mu$ is the grand mean and $\tau_i$ is the effect of group $i$.

### Hypotheses

- $H_0$: $\mu_1 = \mu_2 = \cdots = \mu_k$ (all group means are equal)
- $H_1$: at least one $\mu_i$ differs

### ANOVA Table Logic

ANOVA partitions total variability: $SS_{\text{Total}} = SS_{\text{Between}} + SS_{\text{Within}}$

| Source | SS | df | MS | $F$ |
|---|---|---|---|---|
| Between groups | $SS_B$ | $k-1$ | $MS_B = SS_B/(k-1)$ | $F = MS_B/MS_W$ |
| Within groups (error) | $SS_W$ | $N-k$ | $MS_W = SS_W/(N-k)$ | |
| Total | $SS_T$ | $N-1$ | | |

A large $F$ ratio means variation **between** groups dominates variation **within** groups — evidence against $H_0$.

### Assumptions

1. Independent observations within and across groups
2. Approximately normal distributions within each group
3. Equal variances across groups (homoscedasticity)

In [ ]:
# Simulate three groups with different means
np.random.seed(5)
g1 = np.random.normal(50, 8, 40)
g2 = np.random.normal(55, 8, 40)
g3 = np.random.normal(60, 8, 40)

df_anova = pd.DataFrame({
    "score": np.concatenate([g1, g2, g3]),
    "group": ["A"] * 40 + ["B"] * 40 + ["C"] * 40
})

# Summary statistics per group
print("Group summary statistics:")
print(df_anova.groupby("group")["score"].agg(["mean", "std", "count"]).round(2))

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.boxplot(data=df_anova, x="group", y="score", palette="pastel", hue="group", legend=False, ax=axes[0])
axes[0].set_title("Boxplots by Group")

for grp, color in zip(["A", "B", "C"], ["steelblue", "orange", "green"]):
    subset = df_anova[df_anova["group"] == grp]["score"]
    axes[1].hist(subset, bins=12, alpha=0.5, color=color, label=f"Group {grp}")
axes[1].set_title("Overlaid Histograms")
axes[1].set_xlabel("Score")
axes[1].legend()

plt.suptitle("One-Way ANOVA: Three Groups", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Fit one-way ANOVA using statsmodels
anova_model = smf.ols("score ~ C(group)", data=df_anova).fit()
anova_table = sm.stats.anova_lm(anova_model, typ=1)

print("ANOVA Table (Type I SS):")
print(anova_table.round(4))
print()

p_anova = anova_table.loc["C(group)", "PR(>F)"]
print("Decision at α = 0.05:",
      "Reject H₀ — at least one group mean differs." if p_anova < 0.05
      else "Fail to reject H₀.")

---
## 23. Multiple Regression

**Multiple regression** extends simple linear regression to **two or more predictors**:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_k x_k + \varepsilon$$

### Interpreting coefficients

$\hat{\beta}_j$ is the expected change in $y$ for a one-unit increase in $x_j$, **holding all other predictors constant**.

### Key output from the regression summary

| Quantity | Meaning |
|---|---|
| $\hat{\beta}_j$ (coef) | Estimated slope for predictor $j$ |
| $t$ (or $p$-value for coef) | Test $H_0: \beta_j = 0$ — is predictor $j$ useful? |
| $R^2$ | Proportion of variance in $y$ explained by all predictors |
| Adjusted $R^2$ | $R^2$ penalised for number of predictors (use for model comparison) |
| $F$-statistic | Tests $H_0: \beta_1 = \beta_2 = \cdots = \beta_k = 0$ (overall model) |

In [ ]:
# Simulate data from y = 5 + 3x₁ − 2x₂ + ε
np.random.seed(42)
n  = 100
x1 = np.random.normal(0, 1, n)
x2 = np.random.normal(0, 1, n)
y  = 5 + 3*x1 - 2*x2 + np.random.normal(0, 2, n)

df_mr = pd.DataFrame({"y": y, "x1": x1, "x2": x2})

model_mr = smf.ols("y ~ x1 + x2", data=df_mr).fit()

print("=== Multiple Regression Results ===")
print(f"  True model:  y = 5 + 3·x₁ − 2·x₂ + ε")
print(f"  Fitted:      ŷ = {model_mr.params['Intercept']:.3f}"
      f" + {model_mr.params['x1']:.3f}·x₁"
      f" + {model_mr.params['x2']:.3f}·x₂")
print(f"  R²:          {model_mr.rsquared:.4f}")
print(f"  Adj. R²:     {model_mr.rsquared_adj:.4f}")
print()
print("Full summary:")
print(model_mr.summary())

In [ ]:
# Pairwise scatterplot matrix to visualise relationships among variables
# Each off-diagonal panel shows the relationship between a pair of variables.

g = sns.pairplot(df_mr, kind="reg",
                 plot_kws={"line_kws": {"color": "red"},
                           "scatter_kws": {"alpha": 0.4}})
g.fig.suptitle("Pairplot of $y$, $x_1$, $x_2$", y=1.02, fontsize=13)
plt.show()

In [ ]:
# Regression diagnostics: residuals vs fitted and Q-Q plot
from scipy.stats import probplot

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Residuals vs fitted
axes[0].scatter(model_mr.fittedvalues, model_mr.resid, alpha=0.6, color="steelblue")
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("Residuals vs Fitted")
axes[0].set_xlabel("Fitted $\\hat{y}$")
axes[0].set_ylabel("Residuals")

# Normal Q-Q plot of residuals
(osm, osr), (slope, intercept, _) = probplot(model_mr.resid)
axes[1].scatter(osm, osr, alpha=0.6, color="steelblue")
axes[1].plot(osm, slope * np.array(osm) + intercept,
             color="red", linestyle="--", label="Reference line")
axes[1].set_title("Normal Q-Q Plot of Residuals")
axes[1].set_xlabel("Theoretical quantiles")
axes[1].set_ylabel("Sample quantiles")
axes[1].legend()

plt.suptitle("Multiple Regression Diagnostics", fontsize=13)
plt.tight_layout()
plt.show()

print("Tip: Residuals vs Fitted should show no pattern (random scatter around 0).")
print("     Q-Q plot points should lie close to the diagonal if residuals are normal.")

---
## Summary

### Linear Regression
- Model: $y = \beta_0 + \beta_1 x + \varepsilon$; OLS minimises $\sum(y_i - \hat{y}_i)^2$.
- **Pearson $r$**: measures linear association; $r^2 = R^2$.
- **Spearman $r_s$**: rank-based; robust to outliers and non-linearity.

### ANOVA
- Tests $H_0: \mu_1 = \cdots = \mu_k$ using the $F$ ratio $= MS_{\text{Between}}/MS_{\text{Within}}$.
- Significant $F$ → at least one mean differs; follow up with post-hoc tests (e.g., Tukey HSD) to identify which.

### Multiple Regression
- Model: $y = \beta_0 + \beta_1 x_1 + \cdots + \beta_k x_k + \varepsilon$.
- Each $\hat{\beta}_j$ is interpreted *holding other predictors constant*.
- Use adjusted $R^2$ to compare models with different numbers of predictors.
- Always check diagnostics: residuals vs fitted, Q-Q plot.

---
## Series Complete 🎉

You have now covered the core of an introductory statistics course:

| Part | Topics |
|---|---|
| 1 | Descriptive statistics, graphs, boxplots |
| 2 | Probability, binomial, normal distributions |
| 3 | CLT, confidence intervals, hypothesis testing foundations |
| 4 | Power, p-values, $t$-distribution |
| 5 | Two-sample inference, chi-square, $F$-test |
| 6 | Regression, ANOVA, multiple regression |

**Suggested next steps:** experimental design, non-parametric tests, logistic regression, or time series analysis.